# OpenRouter API: JSON Output, Pydantic Validation, and Weather Tool Calling

This Colab notebook demonstrates four increasingly advanced OpenRouter workflows:

1. **Basic OpenRouter API call**
2. **JSON-mode response**
3. **Structured output using a Pydantic-generated JSON Schema**
4. **Tool calling workflow using a live weather tool**

The tool-calling section shows the complete agent loop:

> User question → LLM selects tool → Python executes tool → tool result returns to LLM → final answer

This notebook uses OpenRouter through the OpenAI-compatible Python SDK.

Official references:

- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenAI SDK integration: https://openrouter.ai/docs/guides/community/openai-sdk
- Structured outputs: https://openrouter.ai/docs/guides/features/structured-outputs

## 1. Install Required Packages

- `openai`: OpenAI-compatible client used with the OpenRouter base URL
- `pydantic`: Defines and validates structured response schemas
- `requests`: Calls the geocoding and weather APIs

In [1]:
%pip install -q -U openai pydantic requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Add the OpenRouter API Key Safely

Do not hard-code API keys directly in a shared notebook.

The key is entered securely with `getpass` and stored only in the current notebook session.

In [2]:
import os
import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
        "Enter your OpenRouter API key: "
    )

print("✅ OpenRouter API key loaded")

✅ OpenRouter API key loaded


## 3. Imports and OpenRouter Client

OpenRouter exposes an OpenAI-compatible endpoint. Therefore, the normal `OpenAI` client can be pointed to:

```text
https://openrouter.ai/api/v1
```

You may replace the model below with another OpenRouter model that supports the required capability.

In [3]:
import json
from typing import Any, Literal

import requests
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, ConfigDict

MODEL_NAME = "openai/gpt-4o-mini"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],

)

print(f"✅ Client initialized with model: {MODEL_NAME}")

✅ Client initialized with model: openai/gpt-4o-mini


In [ ]:
# For groq change the above wherever replacements possible to below code


# !pip install groq
# import os
# from groq import Groq
# from google.colab import userdata
# import getpass

# # Retrieve the API key from Colab secrets
# GROQ_API_KEY = getpass.getpass('GROQ_API_KEY')

# # Initialize the Groq client
# client = Groq(
#     api_key=GROQ_API_KEY,
# )

## 4. Basic OpenRouter API Call

A basic chat completion requires:

- A model name
- A list of messages
- Optional generation settings such as temperature

In [4]:
basic_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI instructor. Explain concepts simply.",
        },
        {
            "role": "user",
            "content": "Explain what an API is in two sentences.",
        },
    ],
    temperature=0.2,
    top_p=0.9,
    max_tokens=300
)

basic_answer = basic_response.choices[0].message.content
print(basic_answer)

An API, or Application Programming Interface, is a set of rules and tools that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information, making it easier to integrate different systems.


## 5. JSON Mode

JSON mode asks the model to return a valid JSON object.

It is useful when an application needs machine-readable output rather than free-form text.

Important distinction:

- **JSON mode** guarantees valid JSON syntax when supported.
- It does **not necessarily enforce an exact field schema**.
- Pydantic can validate the result after generation.

In [5]:
json_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "Return only a valid JSON object. "
                "Include the keys topic, explanation, and examples."
            ),
        },
        {
            "role": "user",
            "content": "Explain cosine similarity with two simple examples.",
        },
    ],
    response_format={"type": "json_object"},
    temperature=0,
)

json_text = json_response.choices[0].message.content
json_data = json.loads(json_text)

print("Raw JSON text:")
print(json_text)

print("\nParsed Python dictionary:")
print(json.dumps(json_data, indent=2))

Raw JSON text:
{
  "topic": "Cosine Similarity",
  "explanation": "Cosine similarity is a metric used to measure how similar two vectors are, regardless of their magnitude. It calculates the cosine of the angle between two non-zero vectors in an inner product space. The value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates orthogonality (no similarity), and -1 indicates opposite directions.",
  "examples": [
    {
      "example": "Example 1",
      "description": "Consider two vectors A = [1, 2, 3] and B = [4, 5, 6]. The cosine similarity can be calculated using the formula: cos(θ) = (A · B) / (||A|| ||B||). Here, A · B = 1*4 + 2*5 + 3*6 = 32, ||A|| = √(1^2 + 2^2 + 3^2) = √14, and ||B|| = √(4^2 + 5^2 + 6^2) = √77. Thus, cosine similarity = 32 / (√14 * √77) ≈ 0.974."
    },
    {
      "example": "Example 2",
      "description": "Consider two vectors C = [1, 0, 0] and D = [0, 1, 0]. The cosine similarity is calculated as follows: C · D = 1*0 + 0*1 + 

## 6. Define a Pydantic Response Model

Pydantic provides:

1. A clear Python schema
2. Automatic JSON Schema generation
3. Runtime validation
4. Type-safe access to fields

The model below describes the exact answer structure expected from the LLM.

In [7]:
class ConceptExplanation(BaseModel):

    model_config = ConfigDict(extra="forbid")
    topic: str = Field(description="Name of the concept being explained")
    summary: str = Field(
        description="A concise explanation in beginner-friendly language"
    )
    difficulty: Literal["beginner", "intermediate", "advanced"]
    key_points: list[str] = Field(
        min_length=2,
        description="Important ideas the learner should remember",
    )
    example: str = Field(description="One practical example")
    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence score between 0 and 1",
    )


print(json.dumps(ConceptExplanation.model_json_schema(), indent=2))

{
  "additionalProperties": false,
  "properties": {
    "topic": {
      "description": "Name of the concept being explained",
      "title": "Topic",
      "type": "string"
    },
    "summary": {
      "description": "A concise explanation in beginner-friendly language",
      "title": "Summary",
      "type": "string"
    },
    "difficulty": {
      "enum": [
        "beginner",
        "intermediate",
        "advanced"
      ],
      "title": "Difficulty",
      "type": "string"
    },
    "key_points": {
      "description": "Important ideas the learner should remember",
      "items": {
        "type": "string"
      },
      "minItems": 2,
      "title": "Key Points",
      "type": "array"
    },
    "example": {
      "description": "One practical example",
      "title": "Example",
      "type": "string"
    },
    "confidence": {
      "description": "Confidence score between 0 and 1",
      "maximum": 1,
      "minimum": 0,
      "title": "Confidence",
      "type": "numb

## 7. Structured Output Using the Pydantic JSON Schema

OpenRouter structured output uses:

```python
response_format={
    "type": "json_schema",
    "json_schema": {
        "name": "...",
        "strict": True,
        "schema": ...
    }
}
```

Pydantic generates the schema, and then validates the model response.

In [8]:
structured_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": ("You are an AI instructor. Follow the supplied JSON schema exactly."),
        },
        {
            "role": "user",
            "content": "Explain LoRA scaling in simple language.",
        },
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "concept_explanation",
            "strict": True,
            "schema": ConceptExplanation.model_json_schema(),
        },
    },
    temperature=0,
)

structured_text = structured_response.choices[0].message.content

try:
    concept = ConceptExplanation.model_validate_json(structured_text)
    print("✅ Response passed Pydantic validation")
    print(concept.model_dump_json(indent=2))
except ValidationError as exc:
    print("❌ Response failed validation")
    print(exc)

✅ Response passed Pydantic validation
{
  "topic": "LoRA Scaling",
  "summary": "LoRA (Low-Rank Adaptation) scaling is a technique used in machine learning to make models more efficient by reducing the number of parameters that need to be trained. It allows large models to adapt to new tasks without needing to retrain the entire model, which saves time and resources.",
  "difficulty": "intermediate",
  "key_points": [
    "LoRA reduces the number of parameters by using low-rank matrices.",
    "It allows for efficient fine-tuning of large models.",
    "LoRA can improve performance on specific tasks without extensive retraining."
  ],
  "example": "For instance, if you have a large language model that has been trained on general text, you can use LoRA to adapt it to understand legal documents better without retraining the whole model from scratch.",
  "confidence": 0.9
}


## 8. Why Use Both Structured Output and Pydantic?

Structured output and Pydantic solve related but different problems:

| Layer | Responsibility |
|---|---|
| OpenRouter structured output | Instructs and constrains the LLM to follow a JSON Schema |
| Pydantic validation | Verifies the received data inside the Python application |
| Python business logic | Decides what to do with the validated object |

Even when schema-constrained generation is enabled, application-side validation is a good safety practice.

# Part 2: Weather Tool-Calling Workflow

The next section implements a simple agent-like workflow without hiding the logic behind an agent framework.

The model receives a weather function definition. It may either:

- Answer directly, or
- Request a call to `get_current_weather`

Python executes the requested function and sends its result back to the model.

## 9. Implement the Weather Tool

This demo uses Open-Weather:


Open-Weather weather API key is required for this classroom demo.

In [20]:
os.environ["OPENWEATHER_API_KEY"] = getpass.getpass("Enter OPENWEATHER_API_KEY: ")

In [28]:
import os
import requests


def get_current_weather(location: str):
    url = "https://api.openweathermap.org/data/2.5/weather"

    response = requests.get(
        url,
        params={
            "q": location,
            "appid": os.environ["OPENWEATHER_API_KEY"],
            "units": "metric"
        },
        timeout=20
    )

    response.raise_for_status()
    data = response.json()

    return {
        "location": data["name"],
        "temperature": data["main"]["temp"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"],
        "condition": data["weather"][0]["description"]
    }

## 10. Test the Weather Function Directly

Testing the Python function separately makes debugging easier before connecting it to the LLM.

In [30]:
weather = get_current_weather("Mysore")
print(weather)

{'location': 'Mysore', 'temperature': 31.95, 'humidity': 32, 'wind_speed': 6.28, 'condition': 'overcast clouds'}


In [31]:
from pydantic import BaseModel, Field


class WeatherToolInput(BaseModel):
    location: str = Field(
        description="Name of the city, for example Kolkata or Paris"
    )

In [32]:
weather_tool_definition = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": (
            "Get the current temperature, humidity, wind speed, "
            "and weather condition for a city using OpenWeather."
        ),
        "parameters": WeatherToolInput.model_json_schema(),
    },
}

print(json.dumps(weather_tool_definition, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_current_weather",
    "description": "Get the current temperature, humidity, wind speed, and weather condition for a city using OpenWeather.",
    "parameters": {
      "properties": {
        "location": {
          "description": "Name of the city, for example Kolkata or Paris",
          "title": "Location",
          "type": "string"
        }
      },
      "required": [
        "location"
      ],
      "title": "WeatherToolInput",
      "type": "object"
    }
  }
}


## 12. Build the Tool Execution Router

A production application may contain several tools.

The router maps the tool name selected by the LLM to the correct Python function.

In [33]:
AVAILABLE_TOOLS = {
    "get_current_weather": get_current_weather,
}


def execute_tool(tool_name: str, raw_arguments: str) -> dict[str, Any]:
    if tool_name not in AVAILABLE_TOOLS:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    if tool_name == "get_current_weather":
        validated_args = WeatherToolInput.model_validate_json(raw_arguments)
        return AVAILABLE_TOOLS[tool_name](**validated_args.model_dump())

    raise ValueError(f"No executor configured for tool: {tool_name}")

## 13. Complete Tool-Calling Workflow

This function demonstrates the complete loop:

1. Send the user question and tool schema to the model.
2. Inspect whether the model requested a tool.
3. Execute every requested tool.
4. Add each tool result to the conversation.
5. Send the updated conversation back to the model.
6. Return the final natural-language answer.

A maximum-round limit prevents an accidental infinite loop.

In [35]:
def ask_with_weather_tool(question: str):

    messages = [
        {
            "role": "system",
            "content": "Use the weather tool for current weather questions."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # First LLM call
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[weather_tool_definition],
        tool_choice="auto",
        temperature=0
    )

    assistant_message = response.choices[0].message

    # If no tool is needed, return the normal answer
    if not assistant_message.tool_calls:
        return assistant_message.content

    # Get tool call details
    tool_call = assistant_message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)

    print("Tool called:", tool_call.function.name)
    print("Arguments:", arguments)

    # Call the weather function
    weather_result = get_current_weather(
        arguments["location"]
    )

    print("Weather result:", weather_result)

    # Add tool call and result to messages
    messages.append(assistant_message)

    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(weather_result)
        }
    )

    # Second LLM call for final answer
    final_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=[weather_tool_definition],
        temperature=0
    )

    return final_response.choices[0].message.content

## 14. Run Weather Workflow Examples

The first prompt should call the tool.

The second prompt usually does not require a tool because it asks a general conceptual question.

In [36]:
answer = ask_with_weather_tool(
    "What is the current weather in Kolkata?"
)

print(answer)

Tool called: get_current_weather
Arguments: {'location': 'Kolkata'}
Weather result: {'location': 'Kolkata', 'temperature': 32.46, 'humidity': 64, 'wind_speed': 2.61, 'condition': 'overcast clouds'}
The current weather in Kolkata is as follows:
- **Temperature:** 32.46°C
- **Humidity:** 64%
- **Wind Speed:** 2.61 m/s
- **Condition:** Overcast clouds


In [37]:
answer = ask_with_weather_tool(
    "Why does humidity sometimes make hot weather feel more uncomfortable?"
)

print("\n==============================")
print("FINAL ANSWER")
print("==============================")
print(answer)


FINAL ANSWER
Humidity can make hot weather feel more uncomfortable due to the way it affects the body's ability to cool itself. Here are the key reasons:

1. **Evaporation of Sweat**: The human body cools itself primarily through the evaporation of sweat. When humidity is high, the air is already saturated with moisture, which slows down the evaporation process. This means that sweat does not evaporate as efficiently, reducing the body's ability to cool down.

2. **Heat Index**: The combination of temperature and humidity is often expressed as the heat index, which indicates how hot it feels to the human body. High humidity can significantly raise the heat index, making it feel much hotter than the actual temperature.

3. **Discomfort and Fatigue**: High humidity can lead to increased discomfort and fatigue. The body has to work harder to regulate its temperature, which can lead to feelings of lethargy and discomfort.

4. **Respiratory Effects**: High humidity can also affect breathin